### Research course submission

**Name :** Abhijith Sreesylesh Babu

**Paper :** Attribution-Based confidence metric for Deep Neural Networks



# Attribution based confidence values

As the complexities of the deep neural networks increases, the performance of the models increases. But this creates a gap in the confidence of the outputs given by the model. We need a good confidence metric to measure the confidence of the model on an output. ABC is an attribution based confidence metric based on Integrated gradients

### ABC algorithm

ABC algorithm calculates the confidence based on how well the model behaves when an important feature is removed. The algorithm creates a number of samples of the input image, each of them having its features (pixels) set to the base value. The probability that a pixel is chosen in decided based on the attribution value of the pixel obtained by integrated gradients.

In [1]:
# importing the required libraries

import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import os
import contextlib
from tqdm import tqdm
import matplotlib.pyplot as plt
import requests
import random

In [2]:
# laoding the vgg19 model

model = models.vgg19(pretrained=True)
with open(os.devnull, 'w') as fnull:
    with contextlib.redirect_stdout(fnull):
        model.eval()

C:\Users\abhij\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\abhij\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
# Three images chosen for demonstration. The images are scaled down to 60*60 for better performance

images = [
    ["images/img1.jpg"],
    ["images/img2.jpg"],
    ["images/img3.jpg"]
]

image_size = 60

In [4]:
# generating integrated gradients for the given image

def generate_integrated_gradients(img, cls, index):
    image = img
    class_number = cls
    image = image.requires_grad_()

    base_image = 0*image # Black image of size same as input.

    integrated_gradients = None
    steps = 10 # Number of steps in the line

    # Calculating gradients at each step in the line
    for i in tqdm(range(steps+1)):
        t = i/steps
        new_image = base_image + (image-base_image) * t # Equation of the line
        new_image.retain_grad()
        class_scores = model(new_image)
        target_score = class_scores[0, class_number]
        model.zero_grad() # setting gradients to zero
        target_score.backward(retain_graph=True)
        if integrated_gradients is None:
            integrated_gradients = new_image.grad.data.mean(dim=0).mean(dim=0)
        else:  
            integrated_gradients += new_image.grad.data.mean(dim=0).mean(dim=0)

    integrated_gradients = integrated_gradients/steps
    integrated_gradients = integrated_gradients*image.mean(dim=0).mean(dim=0)
    return integrated_gradients


In [5]:
# function to load and transform the image

def load_image(im):
    path = im[0]
    image = Image.open(path)
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor()
    ])
    image = transform(image)
    image = image.unsqueeze(0)
    return image

In [6]:
# function to get the class of the image from the model

def prediction(model, img):
    with torch.no_grad():
        pred = model(img)
    return torch.argmax(pred,dim=1).item()

In [7]:
# function to calculate the sampling probabilities

def compute_sampling_probabilities(attributes, image):
    eps = torch.ones_like(image) * 1e-6
    attr = torch.abs(attributes / (image + eps))
    probs = attr / torch.sum(attr.flatten())
    return probs

In [8]:
# function to sample a new image

def sample(image, sampling_probabilities):
    indices = torch.multinomial(sampling_probabilities.flatten(), 1).item()

    sampled_img = image.clone()
    sampled_img = sampled_img.permute(0,2,3,1)
    sampled_img = sampled_img.reshape(-1, 3)
    for i in sampling_probabilities.flatten():
        rand_num = random.uniform(0,0.000004)
        if rand_num < i:
            sampled_img[indices] = 0
    sampled_img[indices] = 0
    sampled_img = sampled_img.reshape(1, image_size, image_size, 3)
    sampled_img = sampled_img.permute(0, 3, 1, 2)
    return sampled_img

In [9]:
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = requests.get(url)
classes_list = response.text.strip().split("\n")

In [10]:
def get_name(cls):
    return classes_list[cls]

In [11]:
# Calculating the confidence of the model for the given image

plt.figure(figsize=(10,5))
for i in range(len(images)):
    image = load_image(images[i])
    prediction_class = prediction(model, image)
    print("predicted class: ", get_name(prediction_class))

    ig_val = generate_integrated_gradients(image, prediction_class, i)

    sampling_probabilities = compute_sampling_probabilities(ig_val,image.mean(dim=0).mean(dim=0))

    num_samples = 100
    correct_count  = 0
    for _ in range(num_samples):
        sampled_image = sample(image, sampling_probabilities)
        sampled_image = sampled_image
        sample_class = prediction(model, sampled_image)
        if sample_class == prediction_class:
            correct_count += 1
        else:
            print("different prediction of sampled image: ", get_name(sample_class))
    print("Correct count: ", correct_count)
    print(f"Confidence: {correct_count/num_samples}")

predicted class:  coral reef


100%|██████████| 11/11 [00:01<00:00,  6.86it/s]


different prediction of sampled image:  green lizard
different prediction of sampled image:  American chameleon
different prediction of sampled image:  axolotl
different prediction of sampled image:  American chameleon
different prediction of sampled image:  American chameleon
different prediction of sampled image:  axolotl
different prediction of sampled image:  hammerhead
different prediction of sampled image:  hammerhead
different prediction of sampled image:  axolotl
different prediction of sampled image:  axolotl
different prediction of sampled image:  hammerhead
different prediction of sampled image:  jellyfish
different prediction of sampled image:  hammerhead
different prediction of sampled image:  hammerhead
different prediction of sampled image:  axolotl
different prediction of sampled image:  American chameleon
different prediction of sampled image:  hammerhead
different prediction of sampled image:  axolotl
different prediction of sampled image:  axolotl
different predictio

100%|██████████| 11/11 [00:01<00:00,  7.24it/s]


Correct count:  100
Confidence: 1.0
predicted class:  cash machine


100%|██████████| 11/11 [00:01<00:00,  7.42it/s]


different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending machine
different prediction of sampled image:  vending 

<Figure size 1000x500 with 0 Axes>